# 01 Ramom Forest & KNN
random forest의 hyperparameter를 설정해봅시다. 준이가 원하는 hyperparameter를 하나 골라봅시다. hyperparameter의 값은 5종류로 설정해봅시다. 그럼 서로 다른 hyperparameter를 가진 random forest 5개가 만들어지겠죠?
knn의 hyperparameter k 값을 각각 다르게 5개의 서로 다른 knn을 만들어봅시다.
이 10개의 모델 중 titanic dataset의 생존 예측 성능이 가장 좋은 건 누구인가요? (전처리 하는 거 잊지 마세요!)

참조 : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Titanic-Dataset.csv')

In [3]:
df['Sex']

0        male
1      female
2      female
3      female
4        male
        ...  
886      male
887    female
888    female
889      male
890      male
Name: Sex, Length: 891, dtype: str

In [4]:
mapping = {'male':0, 'female':1}

In [5]:
df['Sex'].map(mapping)

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    1
889    0
890    0
Name: Sex, Length: 891, dtype: int64

In [6]:
df['Sex'] = df['Sex'].map(mapping)

In [7]:
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)


In [8]:
df['Age']

0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
886    27.0
887    19.0
888     NaN
889    26.0
890    32.0
Name: Age, Length: 891, dtype: float64

In [9]:
df['Age'] = df['Age'].fillna(df['Age'].mode()[0])


In [10]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [11]:
embarked_mapping = {'Q':0, 'S':1, 'C':2}
df['Embarked'].map(embarked_mapping)

0      1
1      2
2      1
3      1
4      1
      ..
886    1
887    1
888    1
889    2
890    0
Name: Embarked, Length: 891, dtype: int64

In [12]:
df['Embarked'] = df['Embarked'].map(embarked_mapping)

In [13]:
import matplotlib.pyplot as plt

In [14]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)

In [15]:
IQR = Q3-Q1

In [16]:
whisker_width = 1.5
lower_whisker = Q1-whisker_width*IQR
upper_whisker = Q3+whisker_width*IQR

In [17]:
df['Fare'] < lower_whisker

0      False
1      False
2      False
3      False
4      False
       ...  
886    False
887    False
888    False
889    False
890    False
Name: Fare, Length: 891, dtype: bool

In [18]:
df['Fare'] > upper_whisker

0      False
1       True
2      False
3      False
4      False
       ...  
886    False
887    False
888    False
889    False
890    False
Name: Fare, Length: 891, dtype: bool

In [19]:
(df['Fare'] < lower_whisker) | (df['Fare'] > upper_whisker)

0      False
1       True
2      False
3      False
4      False
       ...  
886    False
887    False
888    False
889    False
890    False
Name: Fare, Length: 891, dtype: bool

In [20]:
df[(df['Fare'] < lower_whisker) | (df['Fare'] > upper_whisker)]

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
1,1,1,1,38.0,1,0,71.2833,2
27,0,1,0,19.0,3,2,263.0000,1
31,1,1,1,24.0,1,0,146.5208,2
34,0,1,0,28.0,1,0,82.1708,2
52,1,1,1,49.0,1,0,76.7292,2
...,...,...,...,...,...,...,...,...
846,0,3,0,24.0,8,2,69.5500,1
849,1,1,1,24.0,1,0,89.1042,2
856,1,1,1,45.0,1,1,164.8667,1
863,0,3,1,24.0,8,2,69.5500,1


In [21]:
df[(df['Fare'] > lower_whisker) & (df['Fare'] < upper_whisker)]

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,1
2,1,3,1,26.0,0,0,7.9250,1
3,1,1,1,35.0,1,0,53.1000,1
4,0,3,0,35.0,0,0,8.0500,1
5,0,3,0,24.0,0,0,8.4583,0
...,...,...,...,...,...,...,...,...
886,0,2,0,27.0,0,0,13.0000,1
887,1,1,1,19.0,0,0,30.0000,1
888,0,3,1,24.0,1,2,23.4500,1
889,1,1,0,26.0,0,0,30.0000,2


In [22]:
df = df[(df['Fare'] > lower_whisker) & (df['Fare'] < upper_whisker)]

In [23]:
df.reset_index()

,index,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,0,3,0,22.0,1,0,7.2500,1
1,2,1,3,1,26.0,0,0,7.9250,1
2,3,1,1,1,35.0,1,0,53.1000,1
3,4,0,3,0,35.0,0,0,8.0500,1
4,5,0,3,0,24.0,0,0,8.4583,0
...,...,...,...,...,...,...,...,...,...
770,886,0,2,0,27.0,0,0,13.0000,1
771,887,1,1,1,19.0,0,0,30.0000,1
772,888,0,3,1,24.0,1,2,23.4500,1
773,889,1,1,0,26.0,0,0,30.0000,2


In [24]:
df = df.reset_index()

In [25]:
df = df.iloc[:,1:]

In [26]:
from sklearn.preprocessing import StandardScaler
import numpy as np

In [27]:
scaler = StandardScaler()

In [28]:
np.array(df['Fare'])

array([ 7.25  ,  7.925 , 53.1   ,  8.05  ,  8.4583, 51.8625, 21.075 ,
       11.1333, 30.0708, 16.7   , 26.55  ,  8.05  , 31.275 ,  7.8542,
       16.    , 29.125 , 13.    , 18.    ,  7.225 , 26.    , 13.    ,
        8.0292, 35.5   , 21.075 , 31.3875,  7.225 ,  7.8792,  7.8958,
       27.7208,  7.75  , 10.5   , 52.    ,  7.2292,  8.05  , 18.    ,
       11.2417,  9.475 , 21.    ,  7.8958, 41.5792,  7.8792,  8.05  ,
       15.5   ,  7.75  , 21.6792, 17.8   , 39.6875,  7.8   , 26.    ,
       61.9792, 35.5   , 10.5   ,  7.2292, 27.75  , 46.9   ,  7.2292,
       27.9   , 27.7208, 15.2458, 10.5   ,  8.1583,  7.925 ,  8.6625,
       10.5   , 46.9   , 14.4542, 56.4958,  7.65  ,  7.8958,  8.05  ,
       29.    , 12.475 ,  9.    ,  9.5   ,  7.7875, 47.1   , 10.5   ,
       15.85  , 34.375 ,  8.05  ,  8.05  ,  8.05  ,  7.8542, 61.175 ,
       20.575 ,  7.25  ,  8.05  , 34.6542, 63.3583, 23.    , 26.    ,
        7.8958,  7.8958,  8.6542,  7.925 ,  7.8958,  7.65  ,  7.775 ,
        7.8958, 24.1

In [29]:
np.array(df['Fare']).reshape(-1, 1)

array([[ 7.25  ],
       [ 7.925 ],
       [53.1   ],
       [ 8.05  ],
       [ 8.4583],
       [51.8625],
       [21.075 ],
       [11.1333],
       [30.0708],
       [16.7   ],
       [26.55  ],
       [ 8.05  ],
       [31.275 ],
       [ 7.8542],
       [16.    ],
       [29.125 ],
       [13.    ],
       [18.    ],
       [ 7.225 ],
       [26.    ],
       [13.    ],
       [ 8.0292],
       [35.5   ],
       [21.075 ],
       [31.3875],
       [ 7.225 ],
       [ 7.8792],
       [ 7.8958],
       [27.7208],
       [ 7.75  ],
       [10.5   ],
       [52.    ],
       [ 7.2292],
       [ 8.05  ],
       [18.    ],
       [11.2417],
       [ 9.475 ],
       [21.    ],
       [ 7.8958],
       [41.5792],
       [ 7.8792],
       [ 8.05  ],
       [15.5   ],
       [ 7.75  ],
       [21.6792],
       [17.8   ],
       [39.6875],
       [ 7.8   ],
       [26.    ],
       [61.9792],
       [35.5   ],
       [10.5   ],
       [ 7.2292],
       [27.75  ],
       [46.9   ],
       [ 7

In [30]:
scaler.fit_transform(np.array(df['Fare']).reshape(-1, 1))

array([[-7.79117066e-01],
       [-7.29372504e-01],
       [ 2.59982835e+00],
       [-7.20160548e-01],
       [-6.90070616e-01],
       [ 2.50862999e+00],
       [ 2.39725255e-01],
       [-4.92934760e-01],
       [ 9.02676557e-01],
       [-8.26932009e-02],
       [ 6.43208923e-01],
       [-7.20160548e-01],
       [ 9.91420855e-01],
       [-7.34590156e-01],
       [-1.34280154e-01],
       [ 8.32975214e-01],
       [-3.55367095e-01],
       [ 1.31111403e-02],
       [-7.80959457e-01],
       [ 6.02676317e-01],
       [-3.55367095e-01],
       [-7.21693418e-01],
       [ 1.30278496e+00],
       [ 2.39725255e-01],
       [ 9.99711616e-01],
       [-7.80959457e-01],
       [-7.32747765e-01],
       [-7.31524417e-01],
       [ 7.29491786e-01],
       [-7.42269242e-01],
       [-5.39606213e-01],
       [ 2.51876314e+00],
       [-7.80649935e-01],
       [-7.20160548e-01],
       [ 1.31111403e-02],
       [-4.84946151e-01],
       [-6.15144251e-01],
       [ 2.34198082e-01],
       [-7.3

In [31]:
scaled_arr = scaler.fit_transform(np.array(df['Fare']).reshape(-1, 1))

In [32]:
scaled_arr.flatten()

array([-7.79117066e-01, -7.29372504e-01,  2.59982835e+00, -7.20160548e-01,
       -6.90070616e-01,  2.50862999e+00,  2.39725255e-01, -4.92934760e-01,
        9.02676557e-01, -8.26932009e-02,  6.43208923e-01, -7.20160548e-01,
        9.91420855e-01, -7.34590156e-01, -1.34280154e-01,  8.32975214e-01,
       -3.55367095e-01,  1.31111403e-02, -7.80959457e-01,  6.02676317e-01,
       -3.55367095e-01, -7.21693418e-01,  1.30278496e+00,  2.39725255e-01,
        9.99711616e-01, -7.80959457e-01, -7.32747765e-01, -7.31524417e-01,
        7.29491786e-01, -7.42269242e-01, -5.39606213e-01,  2.51876314e+00,
       -7.80649935e-01, -7.20160548e-01,  1.31111403e-02, -4.84946151e-01,
       -6.15144251e-01,  2.34198082e-01, -7.31524417e-01,  1.75079554e+00,
       -7.32747765e-01, -7.20160548e-01, -1.71127977e-01, -7.42269242e-01,
        2.84252165e-01, -1.62798914e-03,  1.61138549e+00, -7.38584460e-01,
        6.02676317e-01,  3.25418674e+00,  1.30278496e+00, -5.39606213e-01,
       -7.80649935e-01,  

In [33]:
pd.Series(scaled_arr.flatten())

0     -0.779117
1     -0.729373
2      2.599828
3     -0.720161
4     -0.690071
         ...   
770   -0.355367
771    0.897459
772    0.414752
773    0.897459
774   -0.742269
Length: 775, dtype: float64

In [34]:
df['Fare'] = pd.Series(scaled_arr.flatten())

In [35]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,-0.779117,1
1,1,3,1,26.0,0,0,-0.729373,1
2,1,1,1,35.0,1,0,2.599828,1
3,0,3,0,35.0,0,0,-0.720161,1
4,0,3,0,24.0,0,0,-0.690071,0
...,...,...,...,...,...,...,...,...
770,0,2,0,27.0,0,0,-0.355367,1
771,1,1,1,19.0,0,0,0.897459,1
772,0,3,1,24.0,1,2,0.414752,1
773,1,1,0,26.0,0,0,0.897459,2


In [36]:
scaled_age = scaler.fit_transform(df[['Age']])

df['Age'] = scaled_age.flatten()

In [37]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,-0.457434,1,0,-0.779117,1
1,1,3,1,-0.147969,0,0,-0.729373,1
2,1,1,1,0.548327,1,0,2.599828,1
3,0,3,0,0.548327,0,0,-0.720161,1
4,0,3,0,-0.302702,0,0,-0.690071,0
...,...,...,...,...,...,...,...,...
770,0,2,0,-0.070603,0,0,-0.355367,1
771,1,1,1,-0.689533,0,0,0.897459,1
772,0,3,1,-0.302702,1,2,0.414752,1
773,1,1,0,-0.147969,0,0,0.897459,2


In [38]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [39]:
from sklearn.model_selection import train_test_split

X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2026
)

In [40]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
from sklearn.ensemble import RandomForestClassifier

In [42]:
from sklearn.ensemble import RandomForestClassifier

rf_ne_10 = RandomForestClassifier(n_estimators=10)
rf_ne_50 = RandomForestClassifier(n_estimators=50)
rf_ne_100 = RandomForestClassifier(n_estimators=100)
rf_ne_200 = RandomForestClassifier(n_estimators=200)
rf_ne_300 = RandomForestClassifier(n_estimators=300)

In [43]:
rf_ne_10.fit(X_train, y_train)
rf_ne_50.fit(X_train, y_train)
rf_ne_100.fit(X_train, y_train)
rf_ne_200.fit(X_train, y_train)
rf_ne_300.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [44]:
print(f"Prediction accuracy of n_estimators=10 is {rf_ne_10.score(X_test, y_test)}")
print(f"Prediction accuracy of n_estimators=50 is {rf_ne_50.score(X_test, y_test)}")
print(f"Prediction accuracy of n_estimators=100 is {rf_ne_100.score(X_test, y_test)}")
print(f"Prediction accuracy of n_estimators=200 is {rf_ne_200.score(X_test, y_test)}")
print(f"Prediction accuracy of n_estimators=300 is {rf_ne_300.score(X_test, y_test)}")

Prediction accuracy of n_estimators=10 is 0.8092783505154639
Prediction accuracy of n_estimators=50 is 0.8247422680412371
Prediction accuracy of n_estimators=100 is 0.8350515463917526
Prediction accuracy of n_estimators=200 is 0.8195876288659794
Prediction accuracy of n_estimators=300 is 0.8195876288659794


In [45]:
from sklearn.neighbors import KNeighborsClassifier

knn1 = KNeighborsClassifier(n_neighbors=1)
knn3 = KNeighborsClassifier(n_neighbors=3)
knn5 = KNeighborsClassifier(n_neighbors=5)
knn7 = KNeighborsClassifier(n_neighbors=7)
knn9 = KNeighborsClassifier(n_neighbors=9)

In [46]:
knn1.fit(X_train, y_train)
knn3.fit(X_train, y_train)
knn5.fit(X_train, y_train)
knn7.fit(X_train, y_train)
knn9.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",9
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [47]:
print(f"Prediction accuracy of k=1 is {knn1.score(X_test, y_test)}")
print(f"Prediction accuracy of k=3 is {knn3.score(X_test, y_test)}")
print(f"Prediction accuracy of k=5 is {knn5.score(X_test, y_test)}")
print(f"Prediction accuracy of k=7 is {knn7.score(X_test, y_test)}")
print(f"Prediction accuracy of k=9 is {knn9.score(X_test, y_test)}")

Prediction accuracy of k=1 is 0.7474226804123711
Prediction accuracy of k=3 is 0.865979381443299
Prediction accuracy of k=5 is 0.8814432989690721
Prediction accuracy of k=7 is 0.9020618556701031
Prediction accuracy of k=9 is 0.8814432989690721


## 10개의 모델 중 KNN의 k=7이 titanic dataset의 생존 예측 성능이 가장 좋습니다